# Vector Stores

En este laboratorio veremos cómo funciona una vectorstore y cómo podemos buscar otras palabras dentro de estas.







## Dependencias


In [1]:
!pip install datasets openai langchain langchain-community faiss-cpu langchain-openai tiktoken

In [7]:
!pip install \
  cohere==5.15.0 \
  umap-learn[plot]==0.5.7 \
  altair==5.5.0 \
  datasets==3.6.0 \
  usearch==2.17.7 \
  np==1.0.2 \
  fastavro==1.10.0 \
  httpx-sse==0.4.0 \
  types-requests==2.32.0.20250328 \
  dill==0.3.8 \
  multiprocess==0.70.16 \
  xxhash==3.5.0 \
  datashader==0.18.1 \
  pyct==0.5.0 \
  fsspec==2025.3.0 \
  langchain-community \
  faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 56.4 MB/s eta 0:00:00


## Cargar documentos en la Vector Store

Nos descargaremos algunos artículos de Wikipedia

Usaremos un dataset de Cohere con artículos de Wikipedia ya pasados por su modelo de Embedding

[Cohere/wikipedia-2023-11-embed-multilingual-v3](https://huggingface.co/datasets/Cohere/wikipedia-2023-11-embed-multilingual-v3)


In [12]:
from langchain.vectorstores import Chroma
from datasets import load_dataset

lang = "simple"
top_k = 5

docs_stream = load_dataset("Cohere/wikipedia-2023-11-embed-multilingual-v3", lang, split="train", streaming=True)

Nos quedaremos solo con el texto de este conjunto de datos.


In [13]:
from langchain.docstore.document import Document

texts = []
max_docs = 10000

for doc in docs_stream:
    texts.append(Document(page_content = doc['text']))
    if len(texts) >= max_docs:
        break

Cargamos el modelo Embedding de OpenAI


In [14]:
from langchain.embeddings.openai import OpenAIEmbeddings
import getpass

api_key = getpass.getpass("Enter your OpenAI API Key:")
embedding = OpenAIEmbeddings(api_key = api_key)

Enter your OpenAI API Key:··········


## Creamos la vector store

In [15]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(texts, embedding)

## Queries

In [17]:
query = "Can you tell me the Andorran population?"
docs = db.similarity_search_with_score(query)
for doc in docs:
  print(doc)
  print("------------------")

(Document(id='9242c2e2-d8a7-4994-a910-dbf71c602544', metadata={}, page_content="There are about 84,000 people living in the country. The capital is Andorra la Vella. It is ruled by a Spanish Bishop and the French President, who both hold the title of Co-Prince. Andorra's government is a parliamentary democracy."), np.float32(0.2957651))
------------------
(Document(id='1d336b4d-e36b-402d-b776-f08ea0c412eb', metadata={}, page_content='The population of Andorra is mostly (90%) Roman Catholic. Their patron saint is Our Lady of Meritxell.'), np.float32(0.31621337))
------------------
(Document(id='15ea4176-30aa-447e-861b-f873672364ee', metadata={}, page_content='Andorra is a rich country mostly because of tourism.  There are about 10.2\xa0million visitors each year.'), np.float32(0.31694555))
------------------
(Document(id='deb23e56-49e5-468e-b7c8-35b4d95b2054', metadata={}, page_content="Andorra doesn't have an Army. France and Spain help to defend Andorra.  The country has a police forc

In [18]:
query = "Who is Alan Turing?"
docs = db.similarity_search_with_score(query)
for doc in docs:
  print(doc)
  print("------------------")

(Document(id='d0d96fc4-19ea-4a84-9eb0-d0cb1e821cfd', metadata={}, page_content='Alan Mathison Turing OBE FRS (London, 23 June 1912 – Wilmslow, Cheshire, 7 June 1954) was an English mathematician and computer scientist. He was born in Maida Vale, London.'), np.float32(0.21449159))
------------------
(Document(id='7b3e7507-ff10-4a6b-9f5c-4cbbf0d9d72a', metadata={}, page_content='Alan was a brilliant mathematician and cryptographer. He became the founder of modern-day computer science and artificial intelligence. He designed a machine at Bletchley Park to break secret Enigma encrypted messages used by the Nazi German war machine to protect sensitive commercial, diplomatic and military communications during World War 2. This made the single biggest contribution to the Allied victory in the war against Nazi Germany. It possibly saved the lives of an estimated 2 million people, and shortened World War II.'), np.float32(0.22104444))
------------------
(Document(id='136e4bd9-2c90-4626-9dee-36c

In [19]:
query = "Who is Javier Milei?"
docs = db.similarity_search_with_score(query)
for doc in docs:
  print(doc)
  print("------------------")

(Document(id='6c195235-d763-4f93-b47c-76b41232d2f8', metadata={}, page_content='Jalal Allakhverdiyev, member of the Academy of Sciences of the Azerbaijan Soviet Socialist Republic (later called the Azerbaijan National Academy of Sciences); Mathematics; died in 2017'), np.float32(0.438458))
------------------
(Document(id='0c75febc-6b7e-4b0a-b2f5-b3c3312a4452', metadata={}, page_content="March 22 - Mijailo Mijailovic is sentenced to life imprisonment for the equivalent of First-degree murder, found guilty of assassination of Sweden's Foreign Minister Anna Lindh, September 10, 2003."), np.float32(0.44768912))
------------------
(Document(id='7b01044a-a586-48b0-966a-14c44fde8b06', metadata={}, page_content='Nicanor Parra, got the Cervantes Prize, the most important literary prize in the Spanish-speaking world'), np.float32(0.45396686))
------------------
(Document(id='09ccd641-9345-4d5d-aeba-f3017b98e8d9', metadata={}, page_content='The Clay Mathematics Institute has said it will give one

In [20]:
query = "Give the population of Argentina"
docs = db.similarity_search_with_score(query)
for doc in docs:
  print(doc)
  print("------------------")

(Document(id='389bbca6-b7bb-4169-a084-f0c029d0825e', metadata={}, page_content='The majority of the Argentineans are descendants of Europeans mainly from Spain, Italy, Germany, Ireland, France, other Europeans countries and Mestizo representing more than 90% of the total population of the country. More than 300,000 Roma gypsies live in Argentina. Since the 1990s, Romanian, Brazilian and Colombian gypsies arrived in Argentina.'), np.float32(0.30113024))
------------------
(Document(id='208d9366-0f13-4737-b2e1-a0484b0312b1', metadata={}, page_content="Argentina is a Christian country. Most of Argentina's people (80 percent) are Roman Catholic. Argentina also has the largest population of Jewish community after Israel and US. Middle Eastern immigrants who were Muslims converted to Catholicism, but there are still Muslims as well."), np.float32(0.3060348))
------------------
(Document(id='d5a7a8f2-370b-4711-8b34-e046a29c154c', metadata={}, page_content='Argentina (officially the Argentine 